# 03 · Analysis and hypotheses

The PRD defines H1–H4 before interpretation. The rules below operationalise those statements without changing their direction. Outcomes describe observed support, not significance or causal effects. This notebook always uses the full v1 population.

H1: Yes overtime rate > No rate. H2: both attrited mean and median tenure lower (Mixed if one is lower). H3: non-increasing satisfaction rates from 1 to 4 (Mixed if endpoints agree with the statement but the sequence is not monotonic). H4: lower attrited income median. Otherwise Not Supported.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'retainwise').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display
from retainwise.data import RAW, PROCESSED, DATA, validate_source, prepare, dictionary, write_version, load_data
from retainwise.analysis import rate_table, distribution, hypothesis_evidence, HYPOTHESES
pd.set_option('display.max_columns', 40)

from retainwise.data import AGE_ORDER, TENURE_ORDER
from retainwise import charts
import json
df = load_data()
evidence = hypothesis_evidence(df)
print(f'Employees={len(df)}, attrited={df.Attrition_Flag.sum()}, retained={(df.Attrition_Flag == 0).sum()}')
print(f'Observed attrition={100*df.Attrition_Flag.mean():.8f}%')

Employees=1470, attrited=237, retained=1233
Observed attrition=16.12244898%


## BR1–BR2 · Organisational and workplace comparisons

Rates use each category’s own denominator. Small groups are flagged; n<10 cells are withheld in the dashboard. Source scales are ordered, but equal distances between ordinal values are not assumed.

In [2]:
for field in ['Department', 'JobRole', 'OverTime', 'BusinessTravel', 'JobSatisfaction', 'EnvironmentSatisfaction', 'JobInvolvement', 'WorkLifeBalance']:
    display(rate_table(df, field))
charts.rates(rate_table(df, 'OverTime', ['No', 'Yes']), 'OverTime', 'Overtime and observed attrition').show(renderer='plotly_mimetype')

,Department,n,attrited,rate_pct,caution
0,Human Resources,63,12.0,19.047619,Descriptive only
1,Research & Development,961,133.0,13.839750,Descriptive only
2,Sales,446,92.0,20.627803,Descriptive only


,JobRole,n,attrited,rate_pct,caution
0,Healthcare Representative,131,9.0,6.870229,Descriptive only
1,Human Resources,52,12.0,23.076923,Descriptive only
2,Laboratory Technician,259,62.0,23.938224,Descriptive only
3,Manager,102,5.0,4.901961,Descriptive only
4,Manufacturing Director,145,10.0,6.896552,Descriptive only
5,Research Director,80,2.0,2.500000,Descriptive only
6,Research Scientist,292,47.0,16.095890,Descriptive only
7,Sales Executive,326,57.0,17.484663,Descriptive only
8,Sales Representative,83,33.0,39.759036,Descriptive only


,OverTime,n,attrited,rate_pct,caution
0,No,1054,110.0,10.436433,Descriptive only
1,Yes,416,127.0,30.528846,Descriptive only


,BusinessTravel,n,attrited,rate_pct,caution
0,Non-Travel,150,12.0,8.000000,Descriptive only
1,Travel_Frequently,277,69.0,24.909747,Descriptive only
2,Travel_Rarely,1043,156.0,14.956855,Descriptive only


,JobSatisfaction,n,attrited,rate_pct,caution
0,1,289,66.0,22.837370,Descriptive only
1,2,280,46.0,16.428571,Descriptive only
2,3,442,73.0,16.515837,Descriptive only
3,4,459,52.0,11.328976,Descriptive only


,EnvironmentSatisfaction,n,attrited,rate_pct,caution
0,1,284,72.0,25.352113,Descriptive only
1,2,287,43.0,14.982578,Descriptive only
2,3,453,62.0,13.686534,Descriptive only
3,4,446,60.0,13.452915,Descriptive only


,JobInvolvement,n,attrited,rate_pct,caution
0,1,83,28.0,33.734940,Descriptive only
1,2,375,71.0,18.933333,Descriptive only
2,3,868,125.0,14.400922,Descriptive only
3,4,144,13.0,9.027778,Descriptive only


,WorkLifeBalance,n,attrited,rate_pct,caution
0,1,80,25.0,31.250000,Descriptive only
1,2,344,58.0,16.860465,Descriptive only
2,3,893,127.0,14.221725,Descriptive only
3,4,153,27.0,17.647059,Descriptive only


## BR3 · Tenure and compensation

Compare both location and spread. Income is in source units. Quartiles use linear interpolation.

In [3]:
for field in ['YearsAtCompany', 'YearsInCurrentRole', 'TotalWorkingYears', 'YearsWithCurrManager', 'MonthlyIncome']:
    print(field)
    display(distribution(df, field))
charts.boxes(df, 'YearsAtCompany', 'Company tenure (years)').show(renderer='plotly_mimetype')
charts.boxes(df, 'MonthlyIncome', 'Monthly income (source units)').show(renderer='plotly_mimetype')
charts.histogram(df, 'YearsAtCompany', 'Company tenure (years)').show(renderer='plotly_mimetype')

YearsAtCompany


,Status,n,mean,median,q1,q3
0,Attrited,237,5.130802,3.0,1.0,7.0
1,Retained,1233,7.369019,6.0,3.0,10.0


YearsInCurrentRole


,Status,n,mean,median,q1,q3
0,Attrited,237,2.902954,2.0,0.0,4.0
1,Retained,1233,4.484185,3.0,2.0,7.0


TotalWorkingYears


,Status,n,mean,median,q1,q3
0,Attrited,237,8.244726,7.0,3.0,10.0
1,Retained,1233,11.862936,10.0,6.0,16.0


YearsWithCurrManager


,Status,n,mean,median,q1,q3
0,Attrited,237,2.852321,2.0,0.0,5.0
1,Retained,1233,4.367397,3.0,2.0,7.0


MonthlyIncome


,Status,n,mean,median,q1,q3
0,Attrited,237,4787.092827,3202.0,2373.0,5916.0
1,Retained,1233,6832.739659,5204.0,3211.0,8834.0


## BR4 · Demographic audit

These comparisons are descriptive audit prompts. They do not establish discrimination or justify targeting. Gender is the source label, not a complete representation or legal definition.

In [4]:
for field, order in [('AgeBand', AGE_ORDER), ('Gender', None), ('MaritalStatus', None)]:
    display(rate_table(df, field, order))

,AgeBand,n,attrited,rate_pct,caution
0,18–29,326,91.0,27.914110,Descriptive only
1,30–39,622,89.0,14.308682,Descriptive only
2,40–49,349,34.0,9.742120,Descriptive only
3,50–60,173,23.0,13.294798,Descriptive only


,Gender,n,attrited,rate_pct,caution
0,Female,588,87.0,14.795918,Descriptive only
1,Male,882,150.0,17.006803,Descriptive only


,MaritalStatus,n,attrited,rate_pct,caution
0,Divorced,327,33.0,10.091743,Descriptive only
1,Married,673,84.0,12.481426,Descriptive only
2,Single,470,120.0,25.531915,Descriptive only


## H1–H4 · Full-population evidence

Keep the statement, rule and result together. Exact values below support the dashboard and README.

In [5]:
for key, statement in HYPOTHESES.items():
    print(key, statement)
    print('Outcome:', evidence[key]['outcome'])
    if key == 'H1':
        print('Percentage-point difference:', evidence[key]['difference_pp'])
    display(pd.DataFrame(evidence[key]['table']))
(PROCESSED / 'hypothesis_evidence_v1.json').write_text(json.dumps(evidence, indent=2), encoding='utf-8')

H1 Employees recorded as working overtime have a higher observed attrition rate than employees not recorded as working overtime.
Outcome: Supported
Percentage-point difference: 20.092413516274995


,OverTime,n,attrited,rate_pct,caution
0,No,1054,110,10.436433,Descriptive only
1,Yes,416,127,30.528846,Descriptive only


H2 Attrited employees have shorter observed company tenure than retained employees.
Outcome: Supported


,Status,n,mean,median,q1,q3
0,Attrited,237,5.130802,3.0,1.0,7.0
1,Retained,1233,7.369019,6.0,3.0,10.0


H3 Lower job-satisfaction categories have higher observed attrition rates than higher job-satisfaction categories.
Outcome: Mixed


,JobSatisfaction,n,attrited,rate_pct,caution
0,1,289,66,22.837370,Descriptive only
1,2,280,46,16.428571,Descriptive only
2,3,442,73,16.515837,Descriptive only
3,4,459,52,11.328976,Descriptive only


H4 Attrited employees have a lower observed median MonthlyIncome than retained employees.
Outcome: Supported


,Status,n,mean,median,q1,q3
0,Attrited,237,4787.092827,3202.0,2373.0,5916.0
1,Retained,1233,6832.739659,5204.0,3211.0,8834.0


1950

## Interpretation and next questions

H1 is Supported: overtime categories differ by about 20.09 percentage points. H2 is Supported: attrited tenure has a median of 3 years versus 6, with a lower mean too. H3 is Mixed: the rates are about 22.84%, 16.43%, 16.52%, 11.33%, so a monotonic claim fails. H4 is Supported: income medians are 3,202 and 5,204 source units.

These are crude associations. Role, level, age and tenure mix may contribute to differences, but this project does not estimate adjusted effects. Prioritise voluntary research into workload, onboarding and employee experience; do not infer individual departure risk, financial savings or intervention effectiveness. The smaller attrited population, absent dates and fictional data limit all conclusions.